## Help Agent Demo for Solvares Products 

### A question answering agent that is an expert knowledge worker for Solvares Products
### To be used by Employes of Solvares Group, to test differnt LLM Models for HelpAgent Solution
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.


In [1]:
# imports
import os
import glob
from dotenv import load_dotenv
import gradio as gr
from pathlib import Path

In [3]:
# imports for langchain
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
)
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [4]:
# price is a factor for our company, so we're going to use a low cost model
OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"

# Open Source Models
code_EpistemeAI = "EpistemeAI/DeepThinkers-Phi4"
CODE_EpistemeAI_URL = ""

code_prithivMLmods = "prithivMLmods/Equuleus-Opus-14B-Exp"
CODE_prithivMLmods_URL = ""

db_name = "vector_db"

In [13]:
# Load environment variables in a file called .env
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

In [6]:
folder = Path("Anleitungen")
pdf_files = list(folder.glob("*.pdf"))

pages = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
   
    async for page in loader.alazy_load():
        pages.append(page)

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total number of chunks: {len(chunks)}")

Total number of chunks: 2347


## A sidenote on Embeddings, and "Auto-Encoding LLMs"

We will be mapping each chunk of text into a Vector that represents the meaning of the text, known as an embedding.

OpenAI offers a model to do this, which we will use by calling their API with some LangChain code.

This model is an example of an "Auto-Encoding LLM" which generates an output given a complete input.
It's different to all the other LLMs we've discussed today, which are known as "Auto-Regressive LLMs", and generate future tokens based only on past context.

Another example of an Auto-Encoding LLMs is BERT from Google. In addition to embedding, Auto-encoding LLMs are often used for classification.

### Sidenote

In week 8 we will return to RAG and vector embeddings, and we will use an open-source vector encoder so that the data never leaves our computer - that's an important consideration when building enterprise systems and the data needs to remain internal.

In [7]:
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk
# Chroma is a popular open source Vector Database based on SQLLite

embeddings = OpenAIEmbeddings()

# If you would rather use the free Vector Embeddings from HuggingFace sentence-transformers
# Then replace embeddings = OpenAIEmbeddings()
# with:
#from langchain.embeddings import HuggingFaceEmbeddings
#embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Delete if already exists

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()


# Create vectorstore
# Ersten Vectorstore mit erstem Batch erstellen
batch_size = 100  # Passe die Größe an deine Token-Grenze an

vectorstore = Chroma.from_documents(
    documents=chunks[:batch_size], 
    embedding=embeddings, 
    persist_directory=db_name
)

# Rest der chunks in Batches hinzufügen
for i in range(batch_size, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Batch {i//batch_size + 1} hinzugefügt ({len(batch)} chunks)")
    
#vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Batch 2 hinzugefügt (100 chunks)
Batch 3 hinzugefügt (100 chunks)
Batch 4 hinzugefügt (100 chunks)
Batch 5 hinzugefügt (100 chunks)
Batch 6 hinzugefügt (100 chunks)
Batch 7 hinzugefügt (100 chunks)
Batch 8 hinzugefügt (100 chunks)
Batch 9 hinzugefügt (100 chunks)
Batch 10 hinzugefügt (100 chunks)
Batch 11 hinzugefügt (100 chunks)
Batch 12 hinzugefügt (100 chunks)
Batch 13 hinzugefügt (100 chunks)
Batch 14 hinzugefügt (100 chunks)
Batch 15 hinzugefügt (100 chunks)
Batch 16 hinzugefügt (100 chunks)
Batch 17 hinzugefügt (100 chunks)
Batch 18 hinzugefügt (100 chunks)
Batch 19 hinzugefügt (100 chunks)
Batch 20 hinzugefügt (100 chunks)
Batch 21 hinzugefügt (100 chunks)
Batch 22 hinzugefügt (100 chunks)
Batch 23 hinzugefügt (100 chunks)
Batch 24 hinzugefügt (47 chunks)
Vectorstore created with 2347 documents


In [8]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 2,347 vectors with 1,536 dimensions in the vector store


## Time to use LangChain to bring it all together

In [9]:
def create_rag_chain(model_choice):
    """Erstellt eine neue RAG Chain mit dem gewählten Modell"""
    global current_chain, current_model
    
    # LLM basierend auf Auswahl erstellen
    if model_choice == "GPT":
        llm = ChatOpenAI(temperature=0.7, model_name=OPENAI_MODEL)
    else:  # Claude
        llm = ChatAnthropic(model=CLAUDE_MODEL, temperature=0.7)
    
    retriever = vectorstore.as_retriever()
    
    # Contextualize question
    contextualize_q_system_prompt = (
        "Given a chat history and the latest user question "
        "which might reference context in the chat history, "
        "formulate a standalone question which can be understood "
        "without the chat history. Do NOT answer the question, just "
        "reformulate it if needed and otherwise return it as is."
    )
    contextualize_q_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", contextualize_q_system_prompt),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
        ]
    )
    history_aware_retriever = create_history_aware_retriever(
        llm, retriever, contextualize_q_prompt
    )
    
    # Answer question
    qa_system_prompt = (
        "You are an Expert in LP2 Software of Städler Logistic and for question-answering tasks. Use "
        "the following pieces of retrieved context to answer the "
        "question. If you don't know the answer, just say that you "
        "don't know. Use three sentences maximum and keep the answer "
        "concise."
        "\n\n"
        "{context}"
    )
    qa_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", qa_system_prompt),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
        ]
    )
    
    question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
    rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
    
    current_chain = rag_chain
    current_model = model_choice
    return rag_chain


## Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [22]:
def chat(message, history):
    """Chat Funktion die die aktuelle Chain verwendet"""
    if current_chain is None:
        return history, ""
    
    result = current_chain.invoke({"input": message, "chat_history": history})
    
    # History aktualisieren
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": result["answer"]})
    
    return history, ""

In [24]:



def on_model_change(model_choice):
    """Wird aufgerufen wenn das Dropdown geändert wird"""
    create_rag_chain(model_choice)
    return []  # Leere Liste leert den Chat


# Initialisiere mit GPT
create_rag_chain("GPT")

# Gradio Interface
with gr.Blocks() as ui:
    gr.Markdown("## Try out different AIs for Solvares Help Agent")
    
    model = gr.Dropdown(
        ["GPT", "Claude"], 
        label="Select model", 
        value="GPT"
    )
    
    # Manueller Chatbot
    chatbot = gr.Chatbot(type="messages", height=500)
    msg = gr.Textbox(
        placeholder="Stelle deine Frage über LP2 Software...",
        label="Nachricht"
    )
        
    # Submit über Enter
    msg.submit(
        fn=chat,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )
    
    # Clear Button
    clear.click(
        fn=lambda: ([], ""),
        outputs=[chatbot, msg]
    )
    
    # Modellwechsel leert Chat automatisch
    model.change(
        fn=on_model_change,
        inputs=[model],
        outputs=[chatbot]
    )

ui.launch(inbrowser=True)

ui.launch(inbrowser=True, quiet=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [27]:
# And in Gradio:

view = gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
